# US 3.5 -- Downstream Model Training

The ultimate goal of CycleGAN translation is not the translated images themselves,
but using them to train a **downstream segmentation model** for USM images.  This
notebook walks through the full pipeline: translate AOI images, transfer labels,
train a segmentation model, and compare with training on real USM data.

## What you will learn

1. How to translate a directory of AOI images to USM
2. Transferring AOI labels to translated USM images
3. Training a segmentation model on translated data
4. Comparing: real USM training vs translated USM training
5. The `udm-epic3 translate` and `udm-epic3 downstream` CLI commands

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic3]"`
- Trained CycleGAN checkpoint

---
## 1. Pipeline Overview

The downstream training pipeline has three stages:

```
Stage 1: Translate                Stage 2: Train               Stage 3: Evaluate
                                                                                  
+----------+   G_AB    +----------+   +----------+   train   +----------+         
| AOI imgs |---------->| Fake USM |   | Fake USM |---------->| Seg      |         
+----------+           | imgs     |   | + AOI    |           | Model    |         
                       +----------+   | labels   |           +-----+----+         
+----------+                          +----------+                 |              
| AOI      |--- copy labels --->                             evaluate on          
| labels   |                                                 real USM test set    
+----------+                                                                      
```

The key insight: AOI labels transfer directly to the translated images because
CycleGAN preserves spatial structure (especially with defect-aware training).

In [ ]:
import torch
from pathlib import Path

from udm_epic3.models.cyclegan import CycleGANModel
from udm_epic3.translation.translate import translate_single, translate_directory

---
## 2. Translating AOI Images to USM

The `translate_single` function translates one image; `translate_directory`
processes an entire folder.

In [ ]:
# Translate a single image
# NOTE: in practice, load a trained checkpoint first
model = CycleGANModel(
    in_channels=3, out_channels=3,
    n_residual_blocks=9, n_discriminator_layers=3,
)
# model.load_state_dict(torch.load("outputs/epic3/checkpoints/epoch_200.pt"))

# Translate a single AOI image
aoi_image = torch.randn(1, 3, 256, 256)  # replace with real image
model.eval()
with torch.no_grad():
    fake_usm = translate_single(model.G_AB, aoi_image)

print(f"Input AOI shape:       {aoi_image.shape}")
print(f"Translated USM shape:  {fake_usm.shape}")

In [ ]:
# Translate an entire directory
# translate_directory(
#     generator=model.G_AB,
#     input_dir=Path("data/epic3/trainA"),
#     output_dir=Path("data/epic3/translated_usm"),
#     image_size=256,
#     device="cuda",
#     batch_size=8,
# )

print("translate_directory() processes all images in a folder:")
print("  - Reads each image from input_dir")
print("  - Applies the generator (G_AB: AOI -> USM)")
print("  - Saves translated images to output_dir")
print("  - Preserves filenames so labels can be matched")

---
## 3. Label Transfer

Because CycleGAN preserves spatial structure, the AOI defect masks can be
directly used as labels for the translated USM images.  No re-annotation
needed!

```
data/epic3/trainA/aoi_001.png      -->  data/epic3/translated_usm/aoi_001.png
data/epic3/masks/aoi_001.png       -->  (same mask applies to translated image)
```

The filenames are preserved by `translate_directory`, so pairing images
with their labels is straightforward.

---
## 4. Training a Segmentation Model

We train the same segmentation architecture on three datasets to compare:

| Training set | Description | Expected performance |
|-------------|-------------|---------------------|
| Real USM + labels | Upper bound (requires USM labels) | Best |
| Translated USM + AOI labels | Our approach (no USM labels needed) | Good |
| AOI images + AOI labels | Lower bound (domain mismatch) | Worst |

In [ ]:
# NOTE: This is a conceptual demo showing the training comparison.
# In practice, each training run takes hours on GPU.

# Results from a typical experiment (replace with your actual results):
results = {
    "Real USM + labels":          {"dice": 0.82, "iou": 0.70},
    "Translated USM + AOI labels": {"dice": 0.75, "iou": 0.61},
    "AOI images + AOI labels":     {"dice": 0.45, "iou": 0.30},
}

print("Segmentation performance on USM test set:")
print(f"{'Training set':<35} {'Dice':>8} {'IoU':>8}")
print("-" * 55)
for name, metrics in results.items():
    print(f"{name:<35} {metrics['dice']:>8.3f} {metrics['iou']:>8.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualise the comparison
names = list(results.keys())
dice_scores = [r["dice"] for r in results.values()]
iou_scores = [r["iou"] for r in results.values()]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, dice_scores, width, label="Dice", color="#2196F3")
bars2 = ax.bar(x + width/2, iou_scores, width, label="IoU", color="#4CAF50")

ax.set_ylabel("Score", fontsize=12)
ax.set_title("Downstream Segmentation: Real vs Translated Training", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")

# Add value labels
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f"{height:.2f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=10)

fig.tight_layout()
plt.show()

---
## 5. CLI Commands

### Translate AOI images

```bash
# Translate all AOI images to USM style
udm-epic3 translate \
    --checkpoint outputs/epic3/checkpoints/epoch_200.pt \
    --input data/epic3/trainA \
    --output data/epic3/translated_usm \
    --direction AB
```

### Train downstream segmentation

```bash
# Train segmentation on translated data
udm-epic3 downstream \
    --images data/epic3/translated_usm \
    --masks data/epic3/masks \
    --config configs/epic3_downstream.yaml
```

The downstream config specifies the segmentation architecture and training
hyperparameters:

```yaml
segmentation:
  backbone: convnext_tiny
  decoder: unet
  epochs: 50
  lr: 1e-4
  batch_size: 8
```

---
## Next steps

| Notebook | Topic |
|----------|------|
| `epic3_06_multisite.ipynb` | Test translation across different manufacturing sites |